In [ ]:
# Outliers

print("Lap time outlier check:")
print(f"  Minimum lap time: {lap_times['milliseconds'].min() / 1000:.1f} seconds")
print(f"  Maximum lap time: {lap_times['milliseconds'].max() / 1000:.1f} seconds")
print(f"  Laps over 5 minutes (300s): {(lap_times['milliseconds'] > 300_000).sum():,}")
print()

# Show the largest outliers to understand the cause
worst = lap_times.nlargest(3, 'milliseconds')[['raceId', 'driverId', 'lap', 'milliseconds']]
worst['seconds'] = worst['milliseconds'] / 1000
print("Top 3 slowest recorded laps:")
print(worst.to_string(index=False))
print()

# remove laps that are outside 80%–150% of the race median - arbitrary
# removes SC laps and extreme outliers
# all laps less than 80% of median and greater tha 150% of median
print("Removing laps that are outside 80%–150% of the race median")
print()
race_median = lap_times.groupby('raceId')['milliseconds'].median().rename('race_med')
lap_times = lap_times.drop(columns=['race_med'], errors='ignore').join(race_median, on='raceId')
lap_times_clean = lap_times[
    (lap_times['milliseconds'] > lap_times['race_med'] * 0.80) &
    (lap_times['milliseconds'] < lap_times['race_med'] * 1.50)
].copy()

removed = len(lap_times) - len(lap_times_clean)
print(f"Laps removed as outliers: {removed:,} ({removed/len(lap_times)*100:.2f}%)")
print(f"Laps kept for analysis:   {len(lap_times_clean):,}")


In [ ]:
# Bump chart: 2023 season position by round
y25 = cs[cs['year'] == 2025].copy()
top8 = y25.groupby('name')['points'].max().nlargest(8).index.tolist()
y25t = y25[y25['name'].isin(top8)]

COLORS_23 = {'Red Bull':'#1E41FF','Mercedes':'#00D2BE','Ferrari':'#DC143C',
             'McLaren':'#FF8000','Aston Martin':'#006F62','Alpine F1 Team':'#0090FF',
             'Williams':'#4169E1','AlphaTauri':'#2B4562'}

fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle('2025 season — constructor championship position by round', fontsize=13)

for team in top8:
    td = y25t[y25t['name'] == team].sort_values('round')
    ax.plot(td['round'], td['position'], 'o-', label=team,
            color=COLORS_23.get(team, '#888'), linewidth=2, markersize=5)

ax.invert_yaxis()   # P1 at the top
ax.set_xlabel('Race round')
ax.set_ylabel('Championship position')
ax.set_yticks(range(1, 9))
ax.set_yticklabels([f'P{i}' for i in range(1, 9)])
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.15)

plt.tight_layout()
plt.show()


In [ ]:
# Driver Head-to-Head Comparison Brocedes Era

year_range = (2013, 2016)
team_races = base[base['year'].between(*year_range)].copy()

ham = team_races[team_races['driver'] == 'Lewis Hamilton'][['raceId','constructorId','positionOrder']]
rsb = team_races[team_races['driver'] == 'Nico Rosberg'][['raceId','constructorId','positionOrder']]

h2h = ham.merge(rsb, on=['raceId','constructorId'], suffixes=('_ham','_rsb')).dropna()
ham_wins = (h2h['positionOrder_ham'] < h2h['positionOrder_rsb']).sum()
rsb_wins = (h2h['positionOrder_ham'] > h2h['positionOrder_rsb']).sum()

print("Hamilton and Rosberg were teammates at Mercedes from 2013 to 2016.")
print(f"Hamilton vs Rosberg (Mercedes, {year_range[0]}–{year_range[1]}):")
print(f"  Hamilton ahead: {ham_wins} races")
print(f"  Rosberg ahead:   {rsb_wins} races")
print(f"  Hamilton win rate: {ham_wins/(ham_wins+rsb_wins)*100:.1f}%")
print()


---
## 9. Lap time trends <a id='lap-times'></a>

**Question:** How have lap times changed through the eras?

**Safety car lap removal:** We remove any lap that is more than 50% slower than the  
race median — these are almost always safety car laps or slow out-laps after a pit stop.  
Without this step, our averages would be distorted.

We then **normalise** lap times to a 1996 baseline so we can compare across circuits 
-

In [ ]:
# Load lap times, merge with race data, and filter outliers > 150% of the median lap time
lt = lap_times.merge(races[['raceId', 'year', 'circuitId']], on='raceId')

race_med = lt.groupby('raceId')['milliseconds'].median().rename('race_med')
lt = lt.drop(columns=['race_med'], errors='ignore').join(race_med, on='raceId')

lt_clean = lt[lt['milliseconds'] < lt['race_med'] * 1.5].copy()

# Aggregate to annual median lap time
annual = (lt_clean
    .groupby(['raceId', 'year'])['milliseconds'].median()
    .reset_index(name='median_lap_ms')
    .groupby('year')['median_lap_ms'].median()
    .reset_index(name='median_lap_ms'))

# Normalize to 1996 baseline
baseline_1996 = annual[annual['year'] == 1996]['median_lap_ms'].iloc[0]
annual['normalised'] = annual['median_lap_ms'] / baseline_1996

fastest = annual.loc[annual['normalised'].idxmin(), 'year']
slowest = annual.loc[annual['normalised'].idxmax(), 'year']
print(f"Baseline (1996): {baseline_1996/1000:.2f}s")
print(f"Fastest: {fastest}")
print(f"Slowest: {slowest}")

# Normalized lap time trend over time
fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle('F1 pace over time', fontsize=13)

ax.plot(annual['year'], annual['normalised'], 'o-', color='steelblue',
        linewidth=2, markersize=5)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, label='1996 baseline')

# Mark major regulation changes
for yr, label in [(2009, 'Slick tyres'), (2014, 'Hybrid'), (2022, 'Ground effect')]:
    ax.axvline(yr, color='gray', linewidth=0.7, linestyle=':', alpha=0.8)
    ax.text(yr + 0.3, annual['normalised'].max() + 0.003,
            label, fontsize=8, color='gray', rotation=90, va='top')

ax.set_xlabel('Season')
ax.set_ylabel('Normalised median lap time (1996 = 1.0)')
ax.set_title('Lower = faster relative to 1996')
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("Lap generally go through waves, with periods of decreasing lap times as the era progresses")
print("then slows are new regulations are introduced")


---
## 10. Qualifying improvement (Q1→Q3) <a id='qualifying'></a>

**Question:** How much do drivers improve their lap time across Q1, Q2, and Q3?

A **negative delta** means the driver got faster (improved). We expect improvement  
because: the track rubbered in (more grip), the driver learned the track, and  
they try progressively harder as the session goes on.

In [ ]:
# Convert time strings to seconds
def parse_time(t):
    if pd.isna(t) or str(t) in ('\\N', ''):
        return np.nan
    try:
        parts = str(t).split(':')
        if len(parts) == 2:
            return float(parts[0]) * 60 + float(parts[1])
        return float(t)
    except:
        return np.nan

qualifying['q1_s'] = qualifying['q1'].apply(parse_time)
qualifying['q2_s'] = qualifying['q2'].apply(parse_time)
qualifying['q3_s'] = qualifying['q3'].apply(parse_time)

# Best time = fastest across all sessions they participated in
qualifying['q_best_s'] = qualifying[['q1_s', 'q2_s', 'q3_s']].min(axis=1)

# Calculate improvements between sessions
# Negative = improved (faster); positive = got slower
qualifying['q1_to_q2'] = qualifying['q2_s'] - qualifying['q1_s']
qualifying['q2_to_q3'] = qualifying['q3_s'] - qualifying['q2_s']
qualifying['q1_to_q3'] = qualifying['q3_s'] - qualifying['q1_s']  

# keep rows with all three session times
q_all = qualifying.dropna(subset=['q1_s', 'q2_s', 'q3_s']).copy()
q_all = q_all.merge(races[['raceId', 'year']], on='raceId')
q_all = q_all.merge(drivers[['driverId', 'forename', 'surname']], on='driverId')
q_all['driver'] = q_all['forename'] + ' ' + q_all['surname']

print(f"Qualifying sessions with all 3 sessions: {len(q_all):,}")
print(f"Overall average Q1→Q3 improvement: {q_all['q1_to_q3'].mean():.3f} seconds")
print("(Negative = faster in Q3 than Q1)")


---
## 11. Lap 1 analysis <a id='lap1'></a>

**Question:** What happens on the first lap — how much does the field reshuffle,  
and which grid positions carry the most crash risk?

The `lap_times` table records each driver's position at the end of every lap.  
Lap 1 shows us where everyone ended up after the standing start and first corner.

We define a **lap 1 DNF** as any retirement where a driver completed 1 lap or fewer.


In [ ]:
# Extract lap 1 positions
lap1 = (lap_times[lap_times['lap'] == 1][['raceId', 'driverId', 'position']]
        .rename(columns={'position': 'lap1_pos'}))

# Join onto our base table
df_l1 = base.merge(lap1, on=['raceId', 'driverId'], how='left')
df_l1['lap1_pos']  = pd.to_numeric(df_l1['lap1_pos'], errors='coerce')
df_l1['laps']      = pd.to_numeric(results.set_index('resultId').reindex(
                      df_l1.index, fill_value=np.nan).get('laps', np.nan), errors='coerce')


laps_col = pd.to_numeric(results['laps'], errors='coerce')
results2  = results.copy()
results2['laps_num'] = laps_col
df_l1 = df_l1.merge(results2[['raceId','driverId','laps']].assign(
    laps=pd.to_numeric(results2['laps'], errors='coerce')),
    on=['raceId','driverId'], how='left', suffixes=('','_r'))

df_l1['lap1_gain'] = df_l1['grid'] - df_l1['lap1_pos']   # positive = gain position
df_l1['lap1_dnf']  = ((df_l1['laps_r'] <= 1) & (df_l1['dnf'] == 1)).astype(int)

print(f"Drivers with lap 1 position data: {df_l1['lap1_pos'].notna().sum():,}")
print(f"Lap 1 DNF rate (all time):  {df_l1['lap1_dnf'].mean()*100:.2f}%")
print(f"Lap 1 DNF rate (2010+):     {df_l1[df_l1['year']>=2010]['lap1_dnf'].mean()*100:.2f}%")
print()
print(f"% of drivers who gained places on lap 1: {(df_l1['lap1_gain']>0).mean()*100:.1f}%")
print(f"% who lost places:                       {(df_l1['lap1_gain']<0).mean()*100:.1f}%")
print(f"% who held their grid position:           {(df_l1['lap1_gain']==0).mean()*100:.1f}%")


In [ ]:
# Chart: grid position → lap 1 DNF risk 
df_l1_mod = df_l1[df_l1['year'] >= 2010].copy()
grid_risk  = (df_l1_mod[df_l1_mod['grid'] <= 20]
              .groupby('grid')['lap1_dnf'].mean() * 100)

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('Lap 1 analysis (2010–2026)', fontsize=13)

# DNF rate by grid slot
ax.bar(grid_risk.index, grid_risk.values,
            color=['#E8593C' if v > grid_risk.mean() else '#4A90D9' for v in grid_risk.values],
            width=0.7)
ax.axhline(grid_risk.mean(), color='gray', linewidth=0.8, linestyle='--',
                label=f'Average ({grid_risk.mean():.1f}%)')
ax.set_xlabel('Starting grid position')
ax.set_ylabel('Lap 1 DNF rate (%)')
ax.set_title('Lap 1 DNF rate by grid slot\n(red = above average risk)')
ax.legend()


plt.tight_layout()
plt.show()

